In [60]:
import pandas as pd
import numpy as np
from EssSimulation import EssSimulationModel
import calendar
import copy

In [61]:
exp_name = "estimate830"
month_num = 9
node_name = "route_B_{:02d}".format(month_num)

In [62]:
es_info = {"transform_capacity": 63000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
           "usable_depth": 0.97,
           "charge_loss": 0.92,
           "discharge_loss": 0.95,
           "es_charge_max": 9000,
           "es_charge_min": -9000,
           "es_capacity_max": 18000,
           "es_capacity_min": 0}

In [63]:
ratio_result_list = []
for ratio in range(10, 240, 10):
    demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
    demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
    demand_load_df.set_index('time', inplace=True)

    strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ratio_experiment_dod97/schedule_result_fixline_up{ratio}.csv")
    strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
    strategy_df['time'] = pd.to_datetime(strategy_df['time'])
    strategy_df.set_index('time', inplace=True)

    ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
    ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
    ele_price_df.set_index('time', inplace=True)

    simulation_model = EssSimulationModel(es_info)
    es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)
    origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)
    ratio_result_list.append((ratio, origin_balance - opt_balance))
    print(f"突破比例{ratio}%, 收益为{origin_balance - opt_balance}")
    # print(round(origin_balance - opt_balance,0))

突破比例10%, 收益为243030.68456666917
突破比例20%, 收益为261898.29072059318
突破比例30%, 收益为280423.82393126376
突破比例40%, 收益为298943.6832180666
突破比例50%, 收益为316888.34437627066
突破比例60%, 收益为333099.9961456666
突破比例70%, 收益为346771.8841646677
突破比例80%, 收益为359380.26453631185
突破比例90%, 收益为371209.28601338714
突破比例100%, 收益为380858.99881041795
突破比例110%, 收益为387794.12739515956
突破比例120%, 收益为392591.5119913826
突破比例130%, 收益为396719.5196238067
突破比例140%, 收益为399313.76616562344
突破比例150%, 收益为401308.56533670984
突破比例160%, 收益为402618.28425108455
突破比例170%, 收益为403676.74883446004
突破比例180%, 收益为404472.3638716396
突破比例190%, 收益为404654.04136584327
突破比例200%, 收益为404576.90969239734
突破比例210%, 收益为404184.8259179862
突破比例220%, 收益为403560.80843055155
突破比例230%, 收益为402778.9133126419


In [64]:
max_ratio_tuple = max(ratio_result_list, key=lambda x: x[1])

In [65]:
max_ratio = max_ratio_tuple[0]
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ratio_experiment_dod97/schedule_result_fixline_up{max_ratio}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)

In [66]:
print("测算方式一  收益：", (origin_balance - opt_balance), "收益占比：", (origin_balance - opt_balance) / origin_balance)

测算方式一  收益： 404654.04136584327 收益占比： 0.07754757389244274


In [67]:
ori_max_demand = demand_load_df["value"].max()
opt_max_demand = total_load_df["total_load"].max()

print("调度后最大需量：", opt_max_demand, "原始最大需量：", ori_max_demand, "需量抬升成本", (opt_max_demand - ori_max_demand) * 38.4)

调度后最大需量： 12737.76 原始最大需量： 10524.0 需量抬升成本 85008.384
